## 1. Setup

In [19]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from tslearn.datasets import UCR_UEA_datasets

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.utils.data import Dataset, DataLoader

from chronos import BaseChronosPipeline

## 2. Load Data

In [20]:
ds = UCR_UEA_datasets()
X_train, y_train_raw, X_test, y_test_raw = ds.load_dataset('LSST')

# tslearn returns string labels — encode to contiguous integers
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)
N_CLASSES = len(le.classes_)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Classes ({N_CLASSES}): {le.classes_}')

X_train: (2459, 36, 6)  y_train: (2459,)
X_test:  (2466, 36, 6)   y_test:  (2466,)
Classes (14): ['15' '16' '42' '52' '53' '6' '62' '64' '65' '67' '88' '90' '92' '95']


## 3. Preprocessing

chronos expects input shape `(n, n_channels, seq_len)`.  
tslearn returns `(n, seq_len, n_channels)` → we transpose in the Dataset.  
We also do instance normalization

In [21]:
# change from (n, 36, 6) to (n, 6, 36)
X_train = np.transpose(X_train, (0, 2, 1))
X_test  = np.transpose(X_test, (0, 2, 1))
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

def zscore_per_sample(X):
    """Z-score each (sample, passband) series over its 36 timesteps."""
    mu  = X.mean(axis=2, keepdims=True) # (n, 6, 1)
    sig = X.std(axis=2, keepdims=True) + 1e-8
    return (X - mu) / sig

X_train_z = zscore_per_sample(X_train)  # (2459, 36, 6)
X_test_z  = zscore_per_sample(X_test)

X_train_z = torch.from_numpy(X_train_z).float()
y_train   = torch.from_numpy(y_train).long()
X_test_z  = torch.from_numpy(X_test_z).float()
y_test    = torch.from_numpy(y_test).long()

X_train: (2459, 6, 36)  y_train: (2459,)
X_test:  (2466, 6, 36)   y_test:  (2466,)


## 4. Utilities

In [22]:
COLORS = [
    "#ca1b46",'#3cb44b','#ffe119','#4363d8','#f58231','#911eb4',
    '#42d4f4','#f032e6','#bfef45','#fabed4','#469990','#dcbeff',
    '#9A6324','#fffac8'
]

results  = {}   # name → {macro_f1, acc}
all_preds = {}  # name → predicted labels on test set


def record(name, y_true, y_pred):
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    acc      = accuracy_score(y_true, y_pred)
    print(f'{name:45s}  macro-F1={macro_f1:.4f}  acc={acc:.4f}')
    results[name]   = {'macro_f1': macro_f1, 'acc': acc}
    all_preds[name] = y_pred
    return macro_f1, acc

## Chronos 2 - Linear probing

Extract fixed representations from the **pre-trained, frozen** `chronos-2-small` backbone, then fit a RF classifier on top.  
This measures how useful the pre-trained features are.

In [29]:
pipeline = BaseChronosPipeline.from_pretrained(
    pretrained_model_name_or_path="autogluon/chronos-2-small",#"amazon/chronos-t5-tiny",
    device_map="cuda",
)

In [30]:
def extract_embeddings(pipeline, X):
    """
    Run frozen model and collect embeddings.

    Returns a list of (n_variates, num_patches + 2, d_model) tensors, one per time series in the batch.
     - n_variates: number of variates in the time series (6 for LSST)
     - num_patches: number of patches the time series is split into (depends on model configuration and context_length)
        - +2: extra tokens for [REG] and masked output patch
     - d_model: embedding dimension of the model (e.g., 512 for chronos-2-small)
     
    """
    embeddings, _ = pipeline.embed(X, batch_size=64) # List of (n_variates, num_patches + 2, d_model) tensors
    return embeddings

def aggregate_embeddings(embeddings, method='reg'):
    """
    Aggregate patch embeddings into a single vector per time series.

    Parameters:
    - embeddings: list of (n_variates, num_patches + 2, d_model) tensors
    - method: how to aggregate the patch embeddings. Options:
        - 'reg': use the [REG] token embedding and average over passband dimension
        - 'mean': average over passband and patch dimensions (excluding [REG] and masked output tokens)
        - 'max': take max over passband and patch dimensions (excluding [REG] and masked output tokens)

    Returns:
    - agg_emb: (n, d_model) array of aggregated embeddings, where n is the number of time series in the batch
    """
    agg_emb = []
    num_patches = embeddings[0].shape[1] - 2
    for emb in embeddings:
        if method == 'reg':
            agg_emb.append(emb[:, num_patches, :].mean(dim=0))  # average over passband dimension
        elif method == 'reg_masked':
            agg_emb.append(emb[:, num_patches:, :].mean(dim=0).flatten())  # average over passband dimension for both [REG] and masked output tokens
        elif method == 'mean':
            agg_emb.append(emb[:, :num_patches, :].mean(dim=(0, 1)))  # average over passband and patch dimensions
        elif method == 'max':
            agg_emb.append(emb[:, :num_patches, :].max(dim=0).values.max(dim=0).values)  # max over passband and patch dimensions
        else:
            raise ValueError(f'Unknown aggregation method: {method}')
    return np.stack(agg_emb)

train_emb = extract_embeddings(pipeline, X_train_z)
test_emb = extract_embeddings(pipeline, X_test_z)

In [ ]:
train_emb_reg = aggregate_embeddings(train_emb, method='reg')
test_emb_reg = aggregate_embeddings(test_emb, method='reg')

scaler = StandardScaler()
train_emb_reg = scaler.fit_transform(train_emb_reg)
test_emb_reg = scaler.transform(test_emb_reg)

rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=2, random_state=42)
rf.fit(train_emb_reg, y_train)
_ = record('RandomForestClassifier [REG]', y_test, rf.predict(test_emb_reg))

Aggregating [reg]: 100%|██████████| 2466/2466 [00:00<00:00, 88222.81it/s]
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.364101340887828e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.5136457177695775e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.707200401161572e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinA

RidgeClassifierCV [REG]                        macro-F1=0.3289  acc=0.3796
RandomForestClassifier [REG]                   macro-F1=0.3068  acc=0.5856


In [ ]:
train_emb_reg_mask = aggregate_embeddings(train_emb, method='reg_masked')
test_emb_reg_mask = aggregate_embeddings(test_emb, method='reg_masked')

rf = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=5, random_state=42)
rf.fit(train_emb_reg_mask, y_train)
_ = record('RandomForestClassifier [REG_MASKED]', y_test, rf.predict(test_emb_reg_mask))

/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.298295773810025e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.538290464597196e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.698012617294523e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.307825

RidgeClassifierCV [REG_MASKED]                 macro-F1=0.3228  acc=0.6014
RandomForestClassifier [REG_MASKED]            macro-F1=0.3254  acc=0.6034


In [10]:
train_emb_reg = aggregate_embeddings(train_emb, method='mean')
test_emb_reg = aggregate_embeddings(test_emb, method='mean')

clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10), cv=5)
clf.fit(train_emb_reg, y_train)
_ = record('RidgeClassifierCV [MEAN]', y_test, clf.predict(test_emb_reg))

rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
rf.fit(train_emb_reg, y_train)
_ = record("Chronos-2-small [MEAN]", y_test, rf.predict(test_emb_reg))

/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.1428594837734636e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.2551525091785152e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.2745100025645115e-08.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/sc/home/nemesio.navarro/miniconda3/envs/dlts/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.253

RidgeClassifierCV [MEAN]                       macro-F1=0.3375  acc=0.6200
Chronos-2-small [MEAN]                         macro-F1=0.3127  acc=0.6002


In [33]:
train_emb_reg = aggregate_embeddings(train_emb, method='max')
test_emb_reg = aggregate_embeddings(test_emb, method='max')

clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10), cv=5)
clf.fit(train_emb_reg, y_train)
_ = record('RidgeClassifierCV [MAX]', y_test, clf.predict(test_emb_reg))

rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
rf.fit(train_emb_reg, y_train)
_ = record("Chronos-2-small [MAX]", y_test, rf.predict(test_emb_reg))

RidgeClassifierCV [MAX]                        macro-F1=0.2890  acc=0.5738
Chronos-2-small [MAX]                          macro-F1=0.2844  acc=0.5633


In [28]:
train_emb_reg = aggregate_embeddings(train_emb, method='reg')
test_emb_reg = aggregate_embeddings(test_emb, method='reg')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=12,
    refit=True,
    verbose=3
)

grid.fit(train_emb_reg, y_train)

Aggregating [reg]: 100%|██████████| 2466/2466 [00:00<00:00, 83782.12it/s]

Fitting 5 folds for each of 72 candidates, totalling 360 fits


[CV 1/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=100;, score=0.310 total time=   4.9s
[CV 5/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=100;, score=0.330 total time=   5.7s
[CV 2/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=100;, score=0.292 total time=   6.5s
[CV 4/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=100;, score=0.350 total time=   6.8s
[CV 3/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=100;, score=0.311 total time=   7.7s
[CV 4/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=200;, score=0.340 total time=  10.1s
[CV 1/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=200;, score=0.295 total time=  11.5s
[CV 5/5] END class_weight=balanced, max_depth=None, min_samples_split=2, n_estimators=200;, score=0.335 total time=  11.6s
[CV 3/5] END cla

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",12
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the

In [32]:
_, _ = record("Chronos-2-small [REG]", y_test, grid.predict(test_emb_reg))

Chronos-2-small [REG]                          macro-F1=0.3567  acc=0.4899


In [12]:
train_emb_reg_mask = aggregate_embeddings(train_emb, method='reg_masked')
test_emb_reg_mask = aggregate_embeddings(test_emb, method='reg_masked')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=12,
    refit=True,
    verbose=3
)

grid.fit(train_emb_reg_mask, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV 1/5] END max_depth=None, min_samples_split=2, n_estimators=100;, score=0.315 total time=   8.9s
[CV 4/5] END max_depth=None, min_samples_split=2, n_estimators=100;, score=0.343 total time=   8.9s
[CV 2/5] END max_depth=None, min_samples_split=2, n_estimators=100;, score=0.311 total time=   9.5s
[CV 5/5] END max_depth=None, min_samples_split=2, n_estimators=100;, score=0.343 total time=   9.7s
[CV 3/5] END max_depth=None, min_samples_split=2, n_estimators=100;, score=0.302 total time=  10.7s
[CV 1/5] END max_depth=None, min_samples_split=2, n_estimators=200;, score=0.319 total time=  16.8s
[CV 2/5] END max_depth=None, min_samples_split=5, n_estimators=100;, score=0.310 total time=   7.5s
[CV 5/5] END max_depth=None, min_samples_split=2, n_estimators=200;, score=0.338 total time=  18.5s
[CV 2/5] END max_depth=None, min_samples_split=2, n_estimators=200;, score=0.315 total time=  19.9s
[CV 1/5] END max_depth=None, min_sampl

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",12
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and 

In [13]:
grid.best_params_

{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=30, min_samples_split=2, class_weight='balanced', random_state=42)
rf.fit(train_emb_reg_mask, y_train)
y_pred_rf = rf.predict(test_emb_reg_mask)
_, _ = record("Chronos-2-small [REG-MASKED]", y_test, rf.predict(test_emb_reg_mask))

Chronos-2-small [REG-MASKED]                   macro-F1=0.3067  acc=0.5807
